# 01 - Exploratory Data Analysis (EDA)

## Anomaly Detection for Reconciliation Variances

This notebook performs comprehensive EDA on reconciliation data to understand:
- Data distributions and quality
- Variance patterns and trends
- Potential anomaly indicators
- Feature candidates for ML models

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from snowflake.snowpark.context import get_active_session

session = get_active_session()
print(f"Connected to Snowflake")

## 1. Data Overview

In [ ]:
%%sql -r data_overview
SELECT 
    COUNT(*) as total_records,
    COUNT(DISTINCT assignment_id) as unique_assignments,
    COUNT(DISTINCT entity_id) as unique_entities,
    COUNT(DISTINCT period_id) as unique_periods,
    MIN(period_end_date) as earliest_period,
    MAX(period_end_date) as latest_period
FROM COCO_LIVE_DB.DBT.RECONCILIATION_360
WHERE is_active = TRUE

## 2. Reconciliation Status Distribution

In [ ]:
%%sql -r status_dist
SELECT 
    reconciliation_status,
    COUNT(*) as count,
    ROUND(AVG(total_abs_variance), 2) as avg_variance,
    ROUND(MAX(total_abs_variance), 2) as max_variance,
    ROUND(SUM(total_abs_variance), 2) as total_variance
FROM COCO_LIVE_DB.DBT.RECONCILIATION_360
WHERE is_active = TRUE
GROUP BY reconciliation_status
ORDER BY count DESC

In [ ]:
status_df = status_dist

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
status_df.plot(kind='bar', x='RECONCILIATION_STATUS', y='COUNT', ax=ax1, color='steelblue', legend=False)
ax1.set_title('Reconciliation Status Distribution', fontsize=12)
ax1.set_xlabel('Status')
ax1.set_ylabel('Count')
ax1.tick_params(axis='x', rotation=45)

ax2 = axes[1]
status_df.plot(kind='bar', x='RECONCILIATION_STATUS', y='AVG_VARIANCE', ax=ax2, color='coral', legend=False)
ax2.set_title('Average Variance by Status', fontsize=12)
ax2.set_xlabel('Status')
ax2.set_ylabel('Avg Variance ($)')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. Variance Distribution Analysis

In [ ]:
%%sql -r variance_buckets
SELECT 
    CASE 
        WHEN total_abs_variance = 0 THEN '0 - Zero'
        WHEN total_abs_variance < 100 THEN '1 - Under $100'
        WHEN total_abs_variance < 1000 THEN '2 - $100-$1K'
        WHEN total_abs_variance < 10000 THEN '3 - $1K-$10K'
        WHEN total_abs_variance < 100000 THEN '4 - $10K-$100K'
        WHEN total_abs_variance < 1000000 THEN '5 - $100K-$1M'
        ELSE '6 - Over $1M'
    END as variance_bucket,
    COUNT(*) as count
FROM COCO_LIVE_DB.DBT.RECONCILIATION_360
WHERE is_active = TRUE
GROUP BY variance_bucket
ORDER BY variance_bucket

## 4. Temporal Variance Trends

In [ ]:
%%sql -r temporal_trends
SELECT 
    period_end_date,
    COUNT(*) as total_records,
    ROUND(AVG(total_abs_variance), 2) as avg_variance,
    SUM(CASE WHEN reconciliation_status = 'High Variance' THEN 1 ELSE 0 END) as high_variance_count,
    ROUND(PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY total_abs_variance), 2) as p95_variance
FROM COCO_LIVE_DB.DBT.RECONCILIATION_360
WHERE is_active = TRUE
GROUP BY period_end_date
ORDER BY period_end_date

## 5. Entity-Level Analysis

In [ ]:
%%sql -r entity_analysis
SELECT 
    entity_name,
    COUNT(*) as assignment_count,
    ROUND(AVG(total_abs_variance), 2) as avg_variance,
    ROUND(SUM(total_abs_variance), 2) as total_variance,
    SUM(CASE WHEN reconciliation_status = 'High Variance' THEN 1 ELSE 0 END) as high_variance_count
FROM COCO_LIVE_DB.DBT.RECONCILIATION_360
WHERE is_active = TRUE
GROUP BY entity_name
ORDER BY total_variance DESC
LIMIT 20

In [ ]:
corr_sql = """
SELECT 
    total_abs_variance,
    gl_bank_difference,
    gl_subledger_difference,
    COALESCE(balance_gl, 0) as balance_gl,
    COALESCE(balance_bank, 0) as balance_bank,
    reconciliation_count,
    hierarchy_depth,
    CASE WHEN is_key_account THEN 1 ELSE 0 END as is_key_account_flag
FROM COCO_LIVE_DB.DBT.RECONCILIATION_360
WHERE is_active = TRUE
LIMIT 50000
"""
corr_df = session.sql(corr_sql).to_pandas()
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f', ax=ax)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
corr_sql = """
SELECT 
    total_abs_variance,
    gl_bank_difference,
    gl_subledger_difference,
    COALESCE(balance_gl, 0) as balance_gl,
    COALESCE(balance_bank, 0) as balance_bank,
    reconciliation_count,
    hierarchy_depth,
    CASE WHEN is_key_account THEN 1 ELSE 0 END as is_key_account_flag
FROM COCO_LIVE_DB.DBT.RECONCILIATION_360
WHERE is_active = TRUE
LIMIT 50000
"""
corr_df = session.sql(corr_sql).to_pandas()
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f', ax=ax)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 7. Potential Anomaly Indicators

In [ ]:
%%sql -r anomaly_indicators
WITH stats AS (
    SELECT 
        AVG(total_abs_variance) as mean_variance,
        STDDEV(total_abs_variance) as std_variance
    FROM COCO_LIVE_DB.DBT.RECONCILIATION_360
    WHERE is_active = TRUE
)
SELECT 
    r.assignment_id,
    r.entity_name,
    r.account_combination,
    r.period_end_date,
    r.total_abs_variance,
    r.reconciliation_status,
    ROUND((r.total_abs_variance - s.mean_variance) / NULLIF(s.std_variance, 0), 2) as z_score
FROM COCO_LIVE_DB.DBT.RECONCILIATION_360 r
CROSS JOIN stats s
WHERE r.is_active = TRUE
ORDER BY z_score DESC
LIMIT 50

## 8. EDA Summary

Key findings from this analysis:
1. **Variance Distribution**: Highly skewed with most records having low variance
2. **High Variance Rate**: Calculate the % of records flagged as high variance
3. **Entity Patterns**: Some entities consistently show higher variances
4. **Temporal Patterns**: Identify any seasonal or trending patterns

**Next Steps**: Proceed to `02_feature_engineering.ipynb` to create ML-ready features